<a href="https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/11_single_step_evals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11 · Unit tests for agents: single-step evals

Lesson 10's evaluator judged the **final answer**. When it fails you learn that something went
wrong, and nothing about where.

A single-step eval isolates **one decision** — did it pick the right tool, extract the right
fields, retrieve the right document — the same way a unit test isolates one function.

**New in this lesson:** `exact_match`, `create_json_match_evaluator`, `create_llm_as_judge`,
prebuilt judge prompts, rubric design, and **calibration** — the step almost everyone skips.

> **Need a key?** You need a LangSmith API key stored in Colab Secrets (🔑 in the left
> sidebar) as `LANGSMITH_API_KEY`, with **"Notebook access" turned on**. If you have not done
> that yet, run **[00 · Setup](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/00_setup.ipynb)** first — it takes 10 minutes and
> checks everything.

In [ ]:
# --- snippet:setup v1 ---
%pip install -qq \
  "deepagents~=0.7.6" \
  "langchain~=1.3.15" \
  "langchain-openai~=1.5.1" \
  "langsmith~=0.11.0" \
  "openevals~=0.2.0"

import os

try:
    from google.colab import userdata

    key = userdata.get("LANGSMITH_API_KEY")
except Exception:  # not on Colab, or secret unavailable
    from getpass import getpass

    key = os.environ.get("LANGSMITH_API_KEY") or getpass("LANGSMITH_API_KEY: ")

os.environ["LANGSMITH_API_KEY"] = key
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "lcw-11-single-step"

# One constant, used everywhere. Models are served by the LangSmith gateway,
# so this key is the only credential the notebook needs.
MODEL = "langsmith:openai/gpt-5.6-luna"
# --- /snippet ---

print("Ready.")

In [ ]:
#@title Synthetic support data (run me) { display-mode: "form" }
# Six orders, eight tickets, one refund policy. Small on purpose: you should be able to
# read the whole dataset and judge the agent's answers yourself.

# --- snippet:support_data v1 ---
ORDERS = [
    {"id": "1042", "customer": "avery@example.com", "item": "Standing desk", "status": "delivered",  "days_ago": 3,  "price": 429.00},
    {"id": "1043", "customer": "jordan@example.com", "item": "Desk lamp",     "status": "delivered",  "days_ago": 45, "price": 39.00},
    {"id": "1044", "customer": "avery@example.com", "item": "Monitor arm",    "status": "in_transit", "days_ago": 1,  "price": 89.00},
    {"id": "1045", "customer": "sam@example.com",   "item": "Office chair",   "status": "delivered",  "days_ago": 10, "price": 249.00},
    {"id": "1046", "customer": "riley@example.com", "item": "Keyboard tray",  "status": "cancelled",  "days_ago": 7,  "price": 59.00},
    {"id": "1047", "customer": "sam@example.com",   "item": "Laptop stand",   "status": "delivered",  "days_ago": 62, "price": 45.00},
]

TICKETS = [
    {"id": "T-1", "order_id": "1042", "text": "Desk arrived with a cracked leg. Photos attached."},
    {"id": "T-2", "order_id": "1043", "text": "Lamp stopped working. Bought it over a month ago."},
    {"id": "T-3", "order_id": "1044", "text": "Where is my monitor arm? Ordered yesterday."},
    {"id": "T-4", "order_id": "1045", "text": "Chair is fine but I ordered the wrong colour. Can I swap?"},
    {"id": "T-5", "order_id": "1046", "text": "I cancelled this but was still charged."},
    {"id": "T-6", "order_id": "1047", "text": "Laptop stand wobbles. Had it two months."},
    {"id": "T-7", "order_id": "1042", "text": "Following up on the cracked desk leg. Any update?"},
    {"id": "T-8", "order_id": "9999", "text": "Order never arrived."},
]

REFUND_POLICY = """
# Refund policy

- Damaged on arrival: full refund or replacement, no time limit. Photos required.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days of delivery. Restocking fee 10%.
- Cancelled orders: refund within 5 business days. Escalate if the customer was charged.
- Refunds above $200 require human approval.
"""
# --- /snippet ---

print(f"{len(ORDERS)} orders, {len(TICKETS)} tickets, {len(REFUND_POLICY.splitlines())} lines of policy")

In [ ]:
# --- snippet:support_agent v1 ---
from langchain_core.tools import tool


@tool
def lookup_order(order_id: str) -> str:
    """Look up a single order by its numeric ID.

    Returns the customer email, item, delivery status, days since order, and price.
    Use this before answering any question about a specific order.
    """
    for order in ORDERS:
        if order["id"] == order_id:
            return (
                f"Order {order['id']}: {order['item']}, ${order['price']:.2f}, "
                f"status={order['status']}, ordered {order['days_ago']} days ago, "
                f"customer={order['customer']}"
            )
    # An error message is an instruction to a reader who cannot see your code.
    return (
        f"No order with ID {order_id!r}. Order IDs are 4 digits (e.g. 1042). "
        f"Ask the customer to re-check the number on their confirmation email."
    )


@tool
def search_tickets(query: str) -> str:
    """Search past support tickets for a keyword.

    Use this to find whether a customer has written in before about the same problem.
    """
    hits = [t for t in TICKETS if query.lower() in t["text"].lower()]
    if not hits:
        return f"No tickets matching {query!r}."
    return "\n".join(f"{t['id']} (order {t['order_id']}): {t['text']}" for t in hits)


@tool
def get_refund_policy() -> str:
    """Return the full refund policy. Consult this before promising any refund."""
    return REFUND_POLICY
# --- /snippet ---

print("3 tools defined")

---

## 1. Deterministic first, always

If a step has one right answer, do not ask a model whether it is right. Model-based evaluators
cost money, add latency, and can be wrong — three problems a comparison operator does not have.

In [ ]:
from openevals import exact_match

print(exact_match(outputs="repair_only", reference_outputs="repair_only"))
print(exact_match(outputs="refund", reference_outputs="repair_only"))

### Structured extraction

Most real single-step evals are field-level rather than whole-string. Suppose one step of the
support agent extracts a structured summary of a ticket.

In [ ]:
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field


class TicketFacts(BaseModel):
    order_id: str = Field(description="The 4-digit order ID mentioned")
    problem: str = Field(description="Category: damage, fault, delivery, billing, or wrong_item")
    days_since_delivery: int = Field(description="Days since delivery, or -1 if not stated")


extractor = init_chat_model(MODEL).with_structured_output(TicketFacts)

TICKET = "Order 1047: the laptop stand wobbles badly. I've had it about two months now."
# `with_structured_output` is typed as returning `dict | BaseModel`, so validate the
# result back into the model. Worth doing anyway: it fails loudly on a malformed response.
extracted = TicketFacts.model_validate(
    extractor.invoke(f"Extract the facts from this support ticket:\n{TICKET}")
)
print(extracted)

In [ ]:
from openevals.json import create_json_match_evaluator

# Score each field independently, then average. `aggregator="all"` would require every
# field to match, which tells you less about where the problem is.
#
# No `model=` here: exact field comparison needs no judge, and passing one is rejected
# unless you also supply a `rubric` naming the keys that need semantic grading.
field_match = create_json_match_evaluator(aggregator="average")

# Returns a LIST of feedback dicts, one per aggregation.
for feedback in field_match(
    outputs=extracted.model_dump(),
    reference_outputs={"order_id": "1047", "problem": "fault", "days_since_delivery": 62},
):
    print(feedback)

Field-level scoring localises the failure. "Extraction scored 0.67" plus a per-field breakdown
tells you `days_since_delivery` is the weak one — which is a bug you can go and fix.

---

## 2. When determinism runs out

Free text has no comparison operator. Here you need a judge — which is another model, and
therefore another thing that can be wrong.

In [ ]:
from openevals import create_llm_as_judge

REFUND_RUBRIC = """
You are grading a customer support reply for POLICY CORRECTNESS only.
Ignore tone, length, and formatting.

<policy>
- Damaged on arrival: full refund or replacement, no time limit.
- Faulty within 30 days of delivery: full refund or replacement.
- Faulty after 30 days: repair only. No refund.
- Wrong item ordered by the customer: exchange within 14 days. Restocking fee 10%.
- Refunds above $200 require human approval.
</policy>

<facts>
{inputs}
</facts>

<reply>
{outputs}
</reply>

Score true only if the reply offers a remedy the policy actually permits for these facts.
Score false if it offers a refund or replacement where the policy allows repair only,
or omits the human-approval requirement for a refund above $200.
"""

policy_judge = create_llm_as_judge(
    prompt=REFUND_RUBRIC,
    feedback_key="policy_correct",
    model=MODEL,
)

CASE = "Order 1047, laptop stand, faulty, delivered 62 days ago."

good = "Since it's been over 30 days since delivery, we can arrange a repair for you."
bad = "I'm sorry about that! I've gone ahead and issued a full refund for you."

print("good reply:", policy_judge(inputs=CASE, outputs=good))
print()
print("bad reply: ", policy_judge(inputs=CASE, outputs=bad))

Note the shape of that rubric. It:

- **names one dimension** (policy correctness) and explicitly excludes others (tone, length)
- **includes the policy inline**, so the judge is not relying on its own recollection
- **says what makes something false**, not just what makes it true

Vague rubrics produce judges that mostly reward confident-sounding text.

---

## 3. Prebuilt judges

`openevals` ships tested prompts for the common dimensions, so you are not writing every rubric
from scratch.

In [ ]:
from openevals.prompts import CORRECTNESS_PROMPT, HALLUCINATION_PROMPT

correctness = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT, feedback_key="correctness", model=MODEL,
)
hallucination = create_llm_as_judge(
    prompt=HALLUCINATION_PROMPT, feedback_key="hallucination", model=MODEL,
)

Q = "How long do I have to return a chair I ordered in the wrong colour?"
REF = "14 days from delivery, with a 10% restocking fee."

grounded = "You have 14 days from delivery to exchange it, and there's a 10% restocking fee."
invented = "You have 90 days to return it, and we'll waive all fees as a courtesy."

print("grounded -> correctness:  ", correctness(inputs=Q, outputs=grounded, reference_outputs=REF))
print("invented -> correctness:  ", correctness(inputs=Q, outputs=invented, reference_outputs=REF))
print("invented -> hallucination:", hallucination(inputs=Q, outputs=invented, context=REF, reference_outputs=REF))

There are around thirty of these, covering correctness, hallucination, conciseness, RAG
groundedness and retrieval relevance, tool selection, PII leakage, prompt injection, and more.
Check for one before you write your own:

```python
from openevals import prompts
print([p for p in dir(prompts) if p.isupper()])
```

---

## 4. Evaluating one decision: tool selection

This is the single-step eval that pays off fastest. Given a question, did the agent reach for
the right tool? It is fast, cheap, deterministic to check, and localises failure precisely.

In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=MODEL,
    tools=[lookup_order, search_tickets, get_refund_policy],
    system_prompt=(
        "You are a customer support agent. Look up the order before answering questions "
        "about it, and check the refund policy before promising any remedy."
    ),
)


def first_tool_called(question: str) -> str | None:
    """Run the agent and report which tool it reached for first."""
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    for m in result["messages"]:
        for tc in getattr(m, "tool_calls", []) or []:
            return tc["name"]
    return None


CASES = [
    ("What is the status of order 1042?", "lookup_order"),
    ("What's our policy on refunds after 30 days?", "get_refund_policy"),
    ("Has anyone complained about a cracked desk before?", "search_tickets"),
]

for question, expected in CASES:
    actual = first_tool_called(question)
    verdict = "PASS" if actual == expected else "FAIL"
    print(f"[{verdict}] {question[:50]:52} expected={expected:18} actual={actual}")

### 🧠 Checkpoint

Why is a tool-selection eval more stable than a final-response eval when you are iterating on
prompts?

<details><summary>Show answer</summary>

Because it has **one right answer and no wording**.

A final-response eval fails for many unrelated reasons: the agent got the policy right but was
too verbose, or phrased it differently, or your judge was having an off day. You end up
debugging the evaluator rather than the agent.

Tool selection is a discrete choice from a small set. `lookup_order` or not. It moves when
behaviour actually changes, and does not move when only phrasing does.

That makes it the fastest signal in your suite — and, in practice, the one that catches
regressions earliest, because a wrong tool choice is almost always upstream of a wrong answer.

</details>

---

## 5. Rubric quality, side by side

Before trusting a judge, look at what a bad rubric does to it.

In [ ]:
vague_judge = create_llm_as_judge(
    prompt="Is this a good customer support reply?\n\n{outputs}",
    feedback_key="vague",
    model=MODEL,
)

REPLIES = {
    "policy-correct, plain": "It's been over 30 days since delivery, so we can arrange a repair.",
    "policy-WRONG, charming": (
        "Oh no, I'm so sorry to hear that! You deserve better. I've processed a full "
        "refund immediately, and please keep the item with our compliments."
    ),
}

for label, reply in REPLIES.items():
    v = vague_judge(outputs=reply)
    s = policy_judge(inputs=CASE, outputs=reply)
    print(f"{label:26} vague={str(v['score']):6} specific={s['score']}")

The vague judge tends to reward the warm, wrong answer. It was never told what "good" means, so
it fell back on what models are trained to like: helpfulness and enthusiasm.

**An unrubriced judge measures charisma.**

---

## 6. Calibration — the step everyone skips

Your judge is a model. Before you let it grade thousands of runs, check it against human labels
on a handful.

In [ ]:
# Hand-labelled cases. In real work you would label 20-50 from production traces.
LABELLED = [
    (CASE, "Over 30 days, so we can offer a repair.", True),
    (CASE, "I've issued you a full refund.", False),
    (CASE, "We'll replace it free of charge.", False),
    (CASE, "That's out of the refund window, but we can repair it at no cost.", True),
    ("Order 1042 damaged on arrival, $429.",
     "Full refund approved and processed immediately.", False),   # needs human approval >$200
    ("Order 1042 damaged on arrival, $429.",
     "You're entitled to a full refund; it needs manager approval as it's over $200.", True),
    ("Order 1045 wrong colour, delivered 10 days ago.",
     "You can exchange it within 14 days; there's a 10% restocking fee.", True),
    ("Order 1045 wrong colour, delivered 10 days ago.",
     "We'll refund you in full, no charge.", False),
]

agree = 0
for facts, reply, human_label in LABELLED:
    judged = bool(policy_judge(inputs=facts, outputs=reply)["score"])
    match = judged == human_label
    agree += match
    if not match:
        print(f"DISAGREE  human={human_label} judge={judged}  {reply[:60]}")

print(f"\nagreement: {agree}/{len(LABELLED)} = {agree / len(LABELLED):.0%}")

### Reading the number

- **Above ~90%** — usable. Spot-check disagreements periodically.
- **70–90%** — tighten the rubric. Look at what it got wrong and add that case as an explicit
  rule or a few-shot example.
- **Below ~70%** — do not deploy it. You would be adding noise and calling it measurement.

The failure mode this prevents is specific and common: **a green dashboard over a broken agent.**
An uncalibrated judge that scores 0.95 feels like proof and is actually just an unexamined
model agreeing with another model.

In [ ]:
# Few-shot examples are the most direct fix for a judge that disagrees with you.
calibrated_judge = create_llm_as_judge(
    prompt=REFUND_RUBRIC,
    feedback_key="policy_correct",
    model=MODEL,
    few_shot_examples=[
        {
            "inputs": "Order 1042 damaged on arrival, $429.",
            "outputs": "Full refund approved and processed immediately.",
            "score": False,
            "reasoning": "Refund is permitted, but above $200 it requires human approval, "
                         "which the reply omits and instead claims it is already processed.",
        },
        {
            "inputs": "Order 1047, faulty, delivered 62 days ago.",
            "outputs": "We'll replace it free of charge.",
            "score": False,
            "reasoning": "After 30 days the policy allows repair only. A replacement is not permitted.",
        },
    ],
)

agree = sum(
    bool(calibrated_judge(inputs=f, outputs=r)["score"]) == label
    for f, r, label in LABELLED
)
print(f"after few-shot calibration: {agree}/{len(LABELLED)} = {agree / len(LABELLED):.0%}")

### 🧠 Checkpoint

One example scored 1.0 from the judge and 0.0 from `exact_match`.

Which one is wrong — and what does that tell you about which evaluator to trust for that field?

<details><summary>Show answer</summary>

Neither is wrong. They are answering different questions, and the mistake was applying both to
the same field.

`exact_match` asks *"is this string identical?"* — correct for `order_id`, `expected_outcome`,
or an enum. The judge asks *"does this mean the right thing?"* — correct for a customer-facing
sentence.

Applying `exact_match` to free text produces failures on every rewording. Applying a judge to an
order ID burns a model call to compare two four-character strings, and can hallucinate a match.

**Pick the evaluator that matches the field's shape.** In a structured extraction, that usually
means different evaluators per field, which is exactly what `create_json_match_evaluator` with
a per-field `rubric` gives you.

</details>

### ✍️ Exercise

Build and calibrate a judge for one dimension of the support agent's output — pick something
other than policy correctness. Good candidates:

- **Tone**: professional, no over-apologising, no emoji
- **Actionability**: does the reply state a concrete next step with a timeframe?
- **Groundedness**: is every factual claim traceable to a tool result?

Then:

1. Write the rubric with an explicit false condition.
2. Hand-label at least 8 replies (write both good and bad ones deliberately).
3. Measure agreement.
4. If it is under 90%, add few-shot examples from the disagreements and measure again.

Report both numbers. The improvement is the point.

<details><summary>Show a solution</summary>

```python
ACTIONABILITY_RUBRIC = """
Grade this customer support reply for ACTIONABILITY only.
Ignore tone, policy correctness, and length.

<reply>
{outputs}
</reply>

Score true only if the reply states BOTH:
  1. a concrete next step (something specific that will happen), AND
  2. who does it or by when.

Score false if it only expresses sympathy, only explains policy, or says
"we'll be in touch" with no timeframe or owner.
"""

judge = create_llm_as_judge(
    prompt=ACTIONABILITY_RUBRIC, feedback_key="actionable", model=MODEL,
)

LABELLED = [
    ("We'll ship a replacement leg today; it arrives in 3-5 business days.", True),
    ("I'm so sorry about this, that must be frustrating.", False),
    ("Our policy allows repair after 30 days.", False),
    ("I've booked a courier collection for Thursday; you'll get an email confirmation.", True),
    ("We'll be in touch soon.", False),
    ("Your refund of $224.10 was approved and reaches your card in 5 business days.", True),
    ("Someone from the team will look into it.", False),
    ("Please send photos of the damage and we'll process the refund within 24 hours.", True),
]

agree = sum(bool(judge(outputs=r)["score"]) == label for r, label in LABELLED)
print(f"baseline agreement: {agree}/{len(LABELLED)} = {agree / len(LABELLED):.0%}")

calibrated = create_llm_as_judge(
    prompt=ACTIONABILITY_RUBRIC,
    feedback_key="actionable",
    model=MODEL,
    few_shot_examples=[
        {"outputs": "We'll be in touch soon.", "score": False,
         "reasoning": "No timeframe and no owner. 'Soon' is not a commitment."},
        {"outputs": "Someone from the team will look into it.", "score": False,
         "reasoning": "No named owner and no date."},
    ],
)

agree2 = sum(bool(calibrated(outputs=r)["score"]) == label for r, label in LABELLED)
print(f"calibrated agreement: {agree2}/{len(LABELLED)} = {agree2 / len(LABELLED):.0%}")
```

</details>

---

## 📌 Key takeaways

- Use a deterministic evaluator whenever the field has one right answer — it is cheaper and cannot be wrong.
- Field-level scoring localises failure; a single whole-output score does not.
- An LLM judge is **a model you also have to evaluate**.
- A rubric needs one named dimension, the reference material inline, and an explicit false condition.
- An unrubriced judge measures charisma — it rewards the warm, wrong answer.
- Calibrate against human labels before trusting a judge. Under ~70% agreement, it is noise.
- Few-shot examples drawn from disagreements are the most direct calibration fix.
- Tool-selection evals are the most stable signal you have while iterating on prompts.

---

## ➡️ Next

**[12 · Integration tests for agents](https://colab.research.google.com/github/langchain-samples/lc-colab-workshops/blob/main/notebooks/12_trajectory_evals.ipynb)**

Single-step evals check individual decisions. But an agent can make every step defensibly and
still take an absurd path — which is what trajectory evals catch.